## 一、匯入套件與cleaned data

In [1]:
!pip install pyarrow


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 導入必要的數據科學套件
# pandas (pd): 用於數據操作、分析和讀取各種數據格式（如CSV、Excel）
# numpy (np): 用於科學計算和數組操作
import pandas as pd
import numpy as np

# 導入繪圖庫
import matplotlib.pyplot as plt  # 用於靜態繪圖
import seaborn as sns  # 基於 Matplotlib 的統計圖形庫
import plotly.express as px  # 用於動態交互式繪圖

In [11]:
# 匯入cleaned data
df_ac = pd.read_parquet('cleaned_data/ac_普通.parquet')
df_putong1 = pd.read_parquet('cleaned_data/putong1_普通新設一.parquet')
df_putong2 = pd.read_parquet('cleaned_data/putong2_普通新設二.parquet')
df_elevator = pd.read_parquet('cleaned_data/elevator_普通電梯.parquet')
df_gongtong = pd.read_parquet('cleaned_data/gongtong_共同教室.parquet')
df_boya1 = pd.read_parquet('cleaned_data/boya1_博雅館一.parquet')
df_boya2 = pd.read_parquet('cleaned_data/boya2_博雅館二.parquet')
df_boya3 = pd.read_parquet('cleaned_data/boya3_博雅三.parquet')
df_boya4 = pd.read_parquet('cleaned_data/boya3_博雅三.parquet')
df_xinsheng = pd.read_parquet('cleaned_data/xinsheng_新生大樓.parquet')
weather = pd.read_csv('cleaned_data/weather_data_2016_2025.csv')
                              

## 二、普通分析

### 1.合併電表與溫度資料

In [14]:
# 合併電表與溫度
putong_all = pd.concat([
    df_ac[['kw']].rename(columns={'kw': 'ac'}),
    df_putong1[['kw']].rename(columns={'kw': 'm1'}),
    df_putong2[['kw']].rename(columns={'kw': 'm2'}),
    df_elevator[['kw']].rename(columns={'kw': 'diantee'}),
], axis=1)
df_weather = weather.set_index('ObsTime')
df_weather.index= pd.to_datetime(df_weather.index)
df_ac.index= pd.to_datetime(df_ac.index)


putong_all_weather = pd.concat([putong_all, df_weather], axis=1)
putong_all_weather

C:\Users\yangs\AppData\Local\Temp\ipykernel_21440\1839162025.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  putong_all_weather = pd.concat([putong_all, df_weather], axis=1)


,ac,m1,m2,diantee,StnPres,SeaPres,Temperature,Td dew point,RH,WS,...,Cloud Amount Sat,TxSoil0cm,TxSoil5cm,TxSoil10cm,TxSoil20cm,TxSoil30cm,TxSoil50cm,TxSoil100cm,WSGust,WDGust
2016-01-01 00:00:00,5.74,6.28,0.14,0.08,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 01:00:00,5.91,7.00,0.15,0.08,1024.6,1028.3,16.7,11.8,73.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.7,60.0
2016-01-01 02:00:00,6.16,4.56,0.14,0.07,1024.1,1027.8,16.6,12.3,76.0,3.7,...,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,70.0
2016-01-01 03:00:00,6.44,5.80,0.14,0.08,1023.7,1027.4,17.0,12.1,73.0,3.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.7,70.0
2016-01-01 04:00:00,6.65,4.43,0.14,0.08,1023.6,1027.3,16.7,12.5,76.0,3.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 21:00:00,4.30,3.15,0.00,0.07,1013.3,1016.9,17.1,16.0,93.0,2.8,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,6.3,86.0
2025-12-31 22:00:00,4.47,1.64,0.00,0.07,1013.4,1017.0,17.2,15.6,90.0,1.6,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,6.7,97.0
2025-12-31 23:00:00,4.88,2.06,0.00,0.07,1013.6,1017.2,16.9,15.6,92.0,0.9,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,4.1,68.0
2025-12-31 23:59:00,NaN,NaN,NaN,NaN,1013.1,1016.7,16.7,15.6,93.0,0.9,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,5.9,94.0


### 2.分出usage type(用電情境)

In [ ]:
# 1. 先取出小時資訊
hour = putong_all_weather.index.hour
temp = putong_all_weather['Temperature']

# 2. 定義四個分類的條件
conditions = [
    # 第一類：冷季凌晨用電 (00:00 - 06:00, temp < 18.9)
    (hour >= 0) & (hour < 6) & (temp < 18.9),
    
    # 第二類：冷季下午用電 (12:00 - 18:00, temp < 20.5)
    (hour >= 12) & (hour < 18) & (temp < 20.5),
    
    # 第三類：暖季凌晨用電 (00:00 - 06:00, temp > 28.5)
    (hour >= 0) & (hour < 6) & (temp > 28.5),
    
    # 第四類：暖季下午用電 (12:00 - 18:00, temp > 31.4)
    (hour >= 12) & (hour < 18) & (temp > 31.4)
]

# 3. 定義對應的名稱
choices = [
    '冷季凌晨用電',
    '冷季下午用電',
    '暖季凌晨用電',
    '暖季下午用電'
]

# 4. 建立新欄位 'usage_type'，若不符合以上條件則標記為 '其他'
putong_all_weather['usage_type'] = np.select(conditions, choices, default='其他')



2016-01-01 00:00:00        其他
2016-01-01 01:00:00    冷季凌晨用電
2016-01-01 02:00:00    冷季凌晨用電
2016-01-01 03:00:00    冷季凌晨用電
2016-01-01 04:00:00    冷季凌晨用電
2016-01-01 05:00:00    冷季凌晨用電
2016-01-01 06:00:00        其他
2016-01-01 07:00:00        其他
2016-01-01 08:00:00        其他
2016-01-01 09:00:00        其他
2016-01-01 10:00:00        其他
2016-01-01 11:00:00        其他
2016-01-01 12:00:00    冷季下午用電
2016-01-01 13:00:00        其他
2016-01-01 14:00:00        其他
2016-01-01 15:00:00        其他
2016-01-01 16:00:00        其他
2016-01-01 17:00:00        其他
2016-01-01 18:00:00        其他
2016-01-01 19:00:00        其他
2016-01-01 20:00:00        其他
2016-01-01 21:00:00        其他
2016-01-01 22:00:00        其他
2016-01-01 23:00:00        其他
2016-01-01 23:59:00        其他
2016-01-02 00:00:00        其他
2016-01-02 01:00:00    冷季凌晨用電
2016-01-02 02:00:00    冷季凌晨用電
2016-01-02 03:00:00    冷季凌晨用電
2016-01-02 04:00:00    冷季凌晨用電
2016-01-02 05:00:00    冷季凌晨用電
2016-01-02 06:00:00        其他
2016-01-02 07:00:00        其他
2016-01-02

### 3.分出日型與情境

In [ ]:
# 假設 df 的 index 已經是 obstime (DatetimeIndex)
# 且具備 'temperature' (溫度) 欄位

# --- 第一步：定義 usage_type (4 類) ---
hour = df.index.hour
temp = df['temperature']

usage_conds = [
    (hour >= 0) & (hour < 6) & (temp < 18.9),   # 1. 冷季凌晨
    (hour >= 12) & (hour < 18) & (temp < 20.5), # 2. 冷季下午
    (hour >= 0) & (hour < 6) & (temp > 28.5),   # 3. 暖季凌晨
    (hour >= 12) & (hour < 18) & (temp > 31.4)  # 4. 暖季下午
]
usage_choices = ['冷季凌晨', '冷季下午', '暖季凌晨', '暖季下午']
df['usage_type'] = np.select(usage_conds, usage_choices, default='其他時段')

# --- 第二步：定義日型 (day_type, 4 類) ---
# 優先順序建議：過年 > 寒暑假 > 例假日 > 平日
month = df.index.month
day = df.index.day
weekday = df.index.weekday # 0=Mon, 6=Sun

# 1. 定義過年 (範例：以 2024 年為例 2/8-2/14，或根據年份動態設定)
# 這裡先示範固定的日期區間，實務上可匯入外部節慶表
is_lunar_new_year = (month == 2) & (day >= 8) & (day <= 14)

# 2. 寒暑假 (台大慣例：1月中-2月中為寒假，6月中-9月中為暑假)
# 簡化版：2月、7月、8月
is_vacation = (month == 2) | (month == 7) | (month == 8)

# 3. 例假日 (週六、週日)
is_weekend = (weekday >= 5)

day_conds = [
    is_lunar_new_year, # 優先判斷過年
    is_vacation,       # 其次判斷寒暑假
    is_weekend         # 再來判斷是否為週末
]
day_choices = ['過年', '寒暑假', '例假日']
df['day_type'] = np.select(day_conds, day_choices, default='平日')

# --- 第三步：交叉結合成 16 類 ---
# 我們只針對非「其他時段」的資料進行 16 類分析
df_analysis = df[df['usage_type'] != '其他時段'].copy()

# 組合兩個指標
df_analysis['category_16'] = df_analysis['day_type'] + '_' + df_analysis['usage_type']

# 檢查 16 類的分布狀況
category_counts = df_analysis['category_16'].value_counts()
print(category_counts)

# 範例輸出：
# 平日_暖季下午    520
# 寒暑假_暖季下午  310
# 例假日_冷季凌晨  150
# ...

### 4.計算分項用電

### 5.組合分項用電